# 1.多LLM链选择示例

知识点讲解：configurable_alternatives 动态替换运行组件

1. 组件定位：运行时组件替换功能
   - 在 LLMOps 项目的应用编排页面中，调试时常常需要"替换大语言模型继续之前的对话"进行调试，
     这就是运行时组件替换功能
   - 它指在构建好的链应用中，动态替换掉特定的模型、提示词等"整个组件本身"，
     而不是替换组件里的参数信息（改参数用 configurable_fields）
   - LangChain 提供 configurable_alternatives() 方法实现，所有 Runnable 组件均支持

2. 典型应用场景
   - 多模型路由：根据任务难度选择 flash（快而便宜）或 pro（强而贵）
   - 多厂商适配：在 OpenAI、文心一言、通义千问之间动态切换
   - 提示词策略切换：为不同用户群体注册不同的 prompt 实现
   - 成本与效果权衡：高价值请求走强模型，普通请求走轻量模型

3. 方法签名与参数含义
   - which: ConfigurableField - 选择器字段，其 id 是运行时的配置键
   - default_key: str - 默认使用的备选项名称，指向调用该方法的原组件
   - **alternatives - 其余备选项，键为选项名，值为对应的 Runnable 实例

4. default_key 的特殊语义
   - default_key 并不新建组件，而是给"调用者自身"起了一个名字
   - 本例中 default_key="deepseek_flash" 指向 ChatOpenAI(model="deepseek-flash")
   - 不传 config 时使用该默认项；传入其他 key 时替换为对应备选组件

5. 底层实现原理
   - 运行流程非常简单：底层通过一个字典 alternatives 存储所有替换组件
   - 从传递的 configurable 字典中获取当前需要选择的组件 key
   - 根据 key 在 alternatives 中找到对应组件返回并执行后续操作
   - 核心代码在 RunnableConfigurableAlternatives._prepare() 方法中：
       which = config.get("configurable", {}).get(self.which.id, self.default_key)
       if which == self.default_key:
           return (self.default, config)
       elif which in self.alternatives:
           alt = self.alternatives[which]
           return (alt, config) if isinstance(alt, Runnable) else (alt(), config)
       else:
           raise ValueError(f"Unknown alternative: {which}")
   - 关键点：提供未注册的 key 会直接抛出 ValueError

6. 组件替换的类型兼容要求
   - 所有备选项必须具备兼容的输入输出类型，才能在同一条链中互换
   - 本例中 ChatOpenAI 与 QianfanChatEndpoint 都接收 Messages、返回 AIMessage
   - 类型不兼容会导致上下游组件报错，需保证接口一致

7. 与 with_fallbacks 的区别（易混淆）
   - configurable_alternatives：主动选择，由调用方通过 config 显式指定
   - with_fallbacks：被动兜底，只有当主组件抛异常时才自动切换（见第 18 章）
   - 两者可组合使用：先按配置选模型，该模型失败时再回退到备用模型

8. 运行流程与数据流转
   - chain.invoke({"query": ...}, config={"configurable": {"llm": "deepseek_pro"}})
   - → 框架读取 configurable.llm = "deepseek_pro"
   - → 从注册表中取出 ChatOpenAI(model="deepseek-v4-pro") 替换链中的 llm 位置
   - → {"query": ...} → prompt → Messages → 被选中的 llm → AIMessage → parser → str

9. 典型输出示例（来自课程文档）
   - 切换到文心一言时，返回内容体现百度模型身份而非默认模型：
       "您好，我是百度研发的知识增强大语言模型，中文名是文心一言，英文名是ERNIE Bot……"
   - 验证点：回复内容随 config 中 llm 的取值变化，说明组件替换在运行时真实生效


In [ ]:
import dotenv
from langchain_community.chat_models.baidu_qianfan_endpoint import QianfanChatEndpoint
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI
dotenv.load_dotenv()
prompt = ChatPromptTemplate.from_template("{query}")
llm = ChatOpenAI(model="deepseek-flash").configurable_alternatives(
    ConfigurableField(id="llm"),
    default_key="deepseek_flash",
    deepseek_pro=ChatOpenAI(model="deepseek-v4-pro"),
    wenxin=QianfanChatEndpoint(),
)
chain = prompt | llm | StrOutputParser()
content = chain.invoke(
    {"query": "你好，你是什么模型呢?"},
    config={"configurable": {"llm": "deepseek_pro"}}
)
print(content)